In [ ]:
%pip install tqdm
%pip install spacy
!python -m spacy download en_core_web_sm
%pip install --no-build-isolation --force-reinstall \
  "numpy==2.3.2" "pandas==2.3.1" "pyarrow==21.0.0"
%pip install --force-reinstall --no-build-isolation \
  "bottleneck>=1.4.0" \
  "numexpr>=2.10.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 18.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
  Obtaining dependency information for numpy==2.3.2 from https://files.pythonhosted.org/packages/b7/13/e792d7209261afb0c9f4759ffef6135b35c77c6349a151f488f531d13595/numpy-2.3.2-cp311-cp311-macosx_14_0_arm64.whl.metadata
  Using cached numpy-2.3.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Obtaining dependency information for pandas==2.3.1 from https://files.pythonhosted.org/packages/ec/d3/3c37cb724d76a841f14b8f5fe57e5e3645207cc67370e4f84717e8bb7657/pandas-2.3.1-cp311-cp311-macosx_11_0_arm64.whl.metadata
  Using cached pandas-2.3.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Obtaining dependency information for pyarrow==21.0.0 from https://files.pythonhosted.org/packages/94/dc/80564a3071a57c20b7c32575e4a0120e8a330ef487c319b122942d665960/pyarrow-21.0.0-cp311-cp311-macosx_12_0_arm64

In [2]:
import pandas as pd
import os
import re
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
import ast
from utils_challenge1 import extraire_contexte_mot, match_tableaux, reconstruire_embeddings
import pickle
from typing import List, Tuple, Dict
import spacy
nlp = spacy.load("en_core_web_sm")
stop_words_spacy = nlp.Defaults.stop_words

In [3]:
matrice_A = np.fromfile("6B.300d.bin", dtype=np.float32).reshape(300, 300)

In [6]:
glove = {}
#on rappelle que le fichier glove ne peut pas être importé sur Git car trop lourd (environ 1go),
#il suffit de le telecharger ici https://www.kaggle.com/datasets/thanakomsn/glove6b300dtxt?resource=download
#puis de le glisser dans le dossier donnees
with open("donnees/glove.6B.300d.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

### si le dataset du Congrès est chargé on peut calculer les datasets sur lesquels on va travailler, mais comme il est trop lourd (4go) pour être importé sur Git, on peut se contenter de charger les datasets de travail du dossier donnees

In [ ]:
#df_imm=extraire_contexte_mot("immigration",rayon=5,supprimer_stopwords=True)
#df_imm=match_tableaux(df_imm)
#df_fed=extraire_contexte_mot("federal_reserve",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_fed=match_tableaux(df_fed)
#df_mon=extraire_contexte_mot("monetary_policy",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_mon=match_tableaux(df_mon)

In [8]:
df_imm=pd.read_csv("donnees/immigration5stopEXCLU.csv")
df_imm=reconstruire_embeddings(df_imm, glove, matrice_A)

In [9]:
df_fed=pd.read_csv("donnees/federal_reserve5stopEXCLU.csv")
df_fed=reconstruire_embeddings(df_fed,glove,matrice_A)

In [10]:
df_mon=pd.read_csv("donnees/monetary_policy5stopEXCLU.csv")
df_mon=reconstruire_embeddings(df_mon,glove,matrice_A)

# Top PPV sur moyenne

In [11]:
def top_voisins_glove_moyennes(
    df,
    glove,
    k: int = 10,
    verbose: bool = True,
    collapse_lemma: bool = False,
):
    """
    Calcule la moyenne des embeddings pour R et D séparément, puis
    renvoie les k voisins GloVe (cosinus) restreints aux mots présents
    dans `contexte_complet` du camp correspondant, après filtrage.

    Paramètres ajoutés:
      - collapse_lemma (bool): si True, regroupe les variantes morphologiques
        (ex: "enact", "enacted", "enacting") en un seul lemme.
    """
    stop_words = set(stop_words_spacy)

    def is_valid_token(w: str) -> bool:
        return (len(w) >= 3) and (re.search(r"[A-Za-z]", w) is not None) and (w not in stop_words)

    word_re = re.compile(r"[A-Za-z]+")

    def lemma_fn_spacy(w: str) -> str:
        return nlp(w)[0].lemma_

    def extract_party_vocab(party_tag: str) -> set:
        vocab = set()
        for txt in df.loc[df["party"] == party_tag, "contexte_complet"]:
            if not isinstance(txt, str):
                continue
            for w in word_re.findall(txt.lower()):
                if is_valid_token(w):
                    vocab.add(w)
        return vocab

    def build_glove_subset(glove_dict: Dict[str, np.ndarray], vocab: set):
        items = [(w, np.asarray(v, dtype=float)) for w, v in glove_dict.items()
                 if (w in vocab) and is_valid_token(w)]
        if not items:
            return [], None
        words = [w for w, _ in items]
        M = np.stack([v for _, v in items], axis=0)
        n = np.linalg.norm(M, axis=1, keepdims=True)
        n[n == 0.0] = 1.0
        return words, M / n

    def mean_emb_for_party(tag: str):
        embs = []
        for e in df.loc[df["party"] == tag, "embedding"]:
            if e is None:
                continue
            arr = np.asarray(e, dtype=float)
            if arr.ndim == 1 and arr.shape[0] == 300 and np.isfinite(arr).all():
                embs.append(arr)
        if not embs:
            return None
        m = np.mean(np.stack(embs, axis=0), axis=0)
        n = np.linalg.norm(m)
        if n == 0.0 or not np.isfinite(n):
            return None
        return m / n

    def top_k_for_party(mean_vec, words, glove_normed):
        if (mean_vec is None) or (glove_normed is None) or (not words):
            return []

        sims = glove_normed @ mean_vec

        if not collapse_lemma:
            kk = min(k, sims.shape[0])
            idxs = np.argpartition(sims, -kk)[-kk:]
            idxs = idxs[np.argsort(sims[idxs])[::-1]]
            return [(words[i], float(sims[i])) for i in idxs]

        # Regroupement par lemme avec spaCy
        groups = {}
        for i, w in enumerate(words):
            lw = lemma_fn_spacy(w)
            s = float(sims[i])
            if lw not in groups or s > groups[lw][1]:
                groups[lw] = (i, s)

        best = sorted(groups.items(), key=lambda kv: kv[1][1], reverse=True)[:k]
        return [(lem, float(score)) for lem, (_idx, score) in best]

    vocab_R = extract_party_vocab("R")
    vocab_D = extract_party_vocab("D")

    words_R, glove_R_normed = build_glove_subset(glove, vocab_R)
    words_D, glove_D_normed = build_glove_subset(glove, vocab_D)

    mean_R = mean_emb_for_party("R")
    mean_D = mean_emb_for_party("D")

    out = {
        "R": top_k_for_party(mean_R, words_R, glove_R_normed),
        "D": top_k_for_party(mean_D, words_D, glove_D_normed),
    }

    if verbose:
        if out["R"]:
            print("🔴 Voisins GloVe (dans le vocabulaire R observé) :")
            for w, s in out["R"]:
                print(f"{w}\t{s:.4f}")
        else:
            print("🔴 Pas de résultat pour R.")

        print()

        if out["D"]:
            print("🔵 Voisins GloVe (dans le vocabulaire D observé) :")
            for w, s in out["D"]:
                print(f"{w}\t{s:.4f}")
        else:
            print("🔵 Pas de résultat pour D.")

    return out

In [12]:
res1=top_voisins_glove_moyennes(df_imm, glove, collapse_lemma=True)

🔴 Voisins GloVe (dans le vocabulaire R observé) :
immigration	0.5655
legislation	0.5472
enact	0.5114
law	0.4952
undocumente	0.4788
enforcement	0.4591
naturalization	0.4542
toughening	0.4532
toughen	0.4434
regulation	0.4306

🔵 Voisins GloVe (dans le vocabulaire D observé) :
legislation	0.5957
immigration	0.5532
enact	0.5433
reform	0.4745
amendment	0.4661
overhaul	0.4660
law	0.4575
enactment	0.4531
provision	0.4492
mandate	0.4448


In [13]:
res2=top_voisins_glove_moyennes(df_fed, glove, collapse_lemma=True)

🔴 Voisins GloVe (dans le vocabulaire R observé) :
reserve	0.6880
greenspan	0.6204
bernanke	0.6013
federal	0.5788
feed	0.5786
policymaker	0.5324
fomc	0.4862
fdic	0.4660
volcker	0.4426
blinder	0.4381

🔵 Voisins GloVe (dans le vocabulaire D observé) :
reserve	0.6860
greenspan	0.6141
bernanke	0.5907
feed	0.5815
federal	0.5740
policymaker	0.5390
fomc	0.4962
fdic	0.4634
volcker	0.4452
blinder	0.4414


In [14]:
res3=top_voisins_glove_moyennes(df_mon, glove, collapse_lemma=True)

🔴 Voisins GloVe (dans le vocabulaire R observé) :
monetary	0.7504
policy	0.6118
expansionary	0.5768
macroeconomic	0.5387
stimulative	0.5052
tighten	0.4991
accommodative	0.4947
budgetary	0.4872
fomc	0.4847
prudent	0.4779

🔵 Voisins GloVe (dans le vocabulaire D observé) :
monetary	0.7464
policy	0.6144
expansionary	0.5725
macroeconomic	0.5258
policymaker	0.5130
tighten	0.5020
stimulative	0.5014
fomc	0.4996
accommodative	0.4915
prudent	0.4724


# Top PPV par occurrences

In [15]:
def top_voisins_compteur(
    df,
    glove,
    top_n: int = 10,
    verbose: bool = True,
    collapse_lemma: bool = False
):
    """
    Pour chaque ligne:
      - extrait les mots de contexte_complet (filtrés)
      - conserve seulement ceux présents dans GloVe
      - choisit le mot dont le vecteur GloVe a la plus grande similarité cosinus
        avec l'embedding de la ligne
      - incrémente +1 ce mot dans un compteur (séparé R / D)

    Args:
        df (pd.DataFrame): colonnes 'contexte_complet' (str), 'embedding' (vec 300), 'party' ('R'/'D')
        glove (dict): {mot(str): np.ndarray(300,)}
        collapse_lemma (bool): si True, regroupe les formes par lemme avec spaCy
    """
    stop_words = set(stop_words_spacy)

    def is_valid_token(w: str) -> bool:
        return (len(w) >= 3) and (re.search(r"[A-Za-z]", w) is not None) and (w not in stop_words)

    word_re = re.compile(r"[A-Za-z]+")

    def lemma_fn_spacy(w: str) -> str:
        return nlp(w)[0].lemma_

    # --- Prépare GloVe filtré + normalisé ---
    glove_normed: Dict[str, np.ndarray] = {}
    for w, v in glove.items():
        if not is_valid_token(w):
            continue
        vec = np.asarray(v, dtype=float)
        if vec.ndim != 1 or vec.shape[0] != 300 or not np.isfinite(vec).all():
            continue
        n = np.linalg.norm(vec)
        if n == 0.0 or not np.isfinite(n):
            continue
        glove_normed[w] = vec / n

    if not glove_normed:
        raise ValueError("Après filtrage, aucun token GloVe valide.")

    counter_R, counter_D = Counter(), Counter()

    # --- Parcours ligne par ligne ---
    for _, row in df.iterrows():
        party = row.get("party")
        if party not in ("R", "D"):
            continue

        emb = row.get("embedding")
        if emb is None:
            continue
        emb = np.asarray(emb, dtype=float)
        if emb.ndim != 1 or emb.shape[0] != 300 or not np.isfinite(emb).all():
            continue
        n = np.linalg.norm(emb)
        if n == 0.0 or not np.isfinite(n):
            continue
        emb = emb / n

        ctx = row.get("contexte_complet")
        if not isinstance(ctx, str) or not ctx:
            continue

        tokens = set(w.lower() for w in word_re.findall(ctx))
        candidates = [w for w in tokens if w in glove_normed]

        if not candidates:
            continue

        C = np.stack([glove_normed[w] for w in candidates])
        sims = C @ emb
        best_idx = int(np.argmax(sims))
        best_word = candidates[best_idx]

        if collapse_lemma:
            best_word = lemma_fn_spacy(best_word)

        if party == "R":
            counter_R.update([best_word])
        else:
            counter_D.update([best_word])

    if verbose:
        print("🔴 Top mots (R) — voisin le plus proche dans le *contexte* :")
        for w, c in counter_R.most_common(top_n):
            print(f"{w}\t{c}")

        print("\n🔵 Top mots (D) — voisin le plus proche dans le *contexte* :")
        for w, c in counter_D.most_common(top_n):
            print(f"{w}\t{c}")

    return counter_R, counter_D

In [16]:
res4=top_voisins_compteur(df_imm, glove, collapse_lemma=True)

🔴 Top mots (R) — voisin le plus proche dans le *contexte* :
illegal	1423
naturalization	1022
bill	792
subcommittee	685
reform	638
law	638
enforcement	628
enforce	602
amendment	523
legislation	407

🔵 Top mots (D) — voisin le plus proche dans le *contexte* :
naturalization	1168
reform	1003
bill	968
subcommittee	783
illegal	695
comprehensive	510
amendment	487
legislation	477
bipartisan	468
enforcement	432


In [17]:
res5=top_voisins_compteur(df_fed, glove, collapse_lemma=True)

🔴 Top mots (R) — voisin le plus proche dans le *contexte* :
reserve	1621
greenspan	613
federal	338
treasury	214
monetary	198
bernanke	173
volcker	161
rate	144
bank	87
chairman	69

🔵 Top mots (D) — voisin le plus proche dans le *contexte* :
reserve	2895
greenspan	742
federal	623
rate	316
treasury	310
monetary	304
volcker	284
bernanke	262
bank	166
chairman	149


In [18]:
res6=top_voisins_compteur(df_mon, glove, collapse_lemma=True)

🔴 Top mots (R) — voisin le plus proche dans le *contexte* :
monetary	822
subcommittee	120
policy	108
rate	32
inflation	27
reserve	24
feed	24
semiannual	19
fiscal	15
oversight	13

🔵 Top mots (D) — voisin le plus proche dans le *contexte* :
monetary	1009
subcommittee	183
policy	145
rate	54
reserve	48
feed	35
fiscal	30
economic	28
inflation	27
semiannual	24


# Top PPV par direction

In [19]:
def top_voisins_direction(
    df: pd.DataFrame,
    glove: dict,
    k: int = 10,
    stop_words=None,
    min_len: int = 3,
    show_classic: bool = False,
    verbose: bool = True,
    collapse_lemma: bool = False
):
    """
    Produit des listes DISTINCTIVES en projetant les mots candidats
    sur la direction d = μ_R − μ_D.

    Paramètres ajoutés :
      - collapse_lemma : si True, regroupe les variantes morphologiques
        par lemme avec spaCy.
    """
    # regex pour extraire des mots composés uniquement de lettres
    _word_re_alpha = re.compile(r"[a-zA-ZÀ-ÿ]+")

    if stop_words is None:
        stop_words = set(stop_words_spacy)
    
    def _is_valid_token(w: str) -> bool:
        return (len(w) >= 3) and (re.search(r"[A-Za-z]", w) is not None) and (w not in stop_words)

    def lemma_fn_spacy(w: str) -> str:
        return nlp(w)[0].lemma_

    def _collect_embeddings(tag: str):
        embs = []
        for e in df.loc[df["party"] == tag, "embedding"]:
            if e is None:
                continue
            a = np.asarray(e, dtype=float)
            if a.ndim == 1 and np.all(np.isfinite(a)):
                embs.append(a)
        if not embs:
            return None
        m = np.mean(np.stack(embs, axis=0), axis=0)
        n = np.linalg.norm(m)
        if not np.isfinite(n) or n == 0:
            return None
        return m / n

    mu_R = _collect_embeddings("R")
    mu_D = _collect_embeddings("D")
    if (mu_R is None) or (mu_D is None):
        if verbose:
            print("Pas assez d'embeddings valides pour R et/ou D.")
        return {"dir_top_R": [], "dir_top_D": []}

    d = mu_R - mu_D
    nd = np.linalg.norm(d)
    if nd == 0 or not np.isfinite(nd):
        if verbose:
            print("μ_R et μ_D sont colinéaires (ou nuls).")
        return {"dir_top_R": [], "dir_top_D": []}
    d = d / nd

    def extract_vocab(tag: str) -> set:
        vocab = set()
        for txt in df.loc[df["party"] == tag, "contexte_complet"]:
            for w in _word_re_alpha.findall(str(txt).lower()):
                if _is_valid_token(w):
                    vocab.add(w)
        return vocab

    vocab_R = extract_vocab("R")
    vocab_D = extract_vocab("D")
    vocab_union = vocab_R | vocab_D

    words = []
    mats = []
    for w in vocab_union:
        v = glove.get(w)
        if v is None:
            continue
        vv = np.asarray(v, dtype=float).ravel()
        if not np.all(np.isfinite(vv)):
            continue
        n = np.linalg.norm(vv)
        if n == 0 or not np.isfinite(n):
            continue
        words.append(w)
        mats.append(vv / n)

    if not words:
        if verbose:
            print("Aucun mot candidat avec vecteur GloVe valide.")
        return {"dir_top_R": [], "dir_top_D": []}

    M = np.stack(mats, axis=0)
    scores = M @ d

    def get_top_k(word_list, score_list, top_k, reverse=False):
        # reverse=False → scores croissants, reverse=True → scores décroissants
        order = np.argsort(score_list)[::-1] if reverse else np.argsort(score_list)
        ordered = [(word_list[i], float(score_list[i])) for i in order]
        if not collapse_lemma:
            return ordered[:top_k]
        # Regroupement par lemme
        groups = {}
        for w, s in ordered:
            lw = lemma_fn_spacy(w)
            if lw not in groups or abs(s) > abs(groups[lw][1]):
                groups[lw] = (w, s)  # garde forme + score le plus extrême
        return sorted(groups.values(), key=lambda x: x[1], reverse=reverse)[:top_k]

    dir_top_R = get_top_k(words, scores, k, reverse=True)
    dir_top_D = get_top_k(words, scores, k, reverse=False)

    out = {"dir_top_R": dir_top_R, "dir_top_D": dir_top_D}

    if show_classic:
        sims_R = M @ mu_R
        sims_D = M @ mu_D
        out["R"] = get_top_k(words, sims_R, k, reverse=True)
        out["D"] = get_top_k(words, sims_D, k, reverse=True)

    if verbose:
        print("🧭 Mots les plus PRO-R (projection sur μ_R − μ_D) :")
        for w, s in dir_top_R:
            print(f"{w}\t{s:.4f}")
        print()
        print("🧭 Mots les plus PRO-D (projection sur μ_R − μ_D) :")
        for w, s in dir_top_D:
            print(f"{w}\t{s:.4f}")
        if show_classic:
            print("\n— Voisins classiques (optionnel) —\n")
            print("🔴 R (cos μ_R):")
            for w, s in out["R"]:
                print(f"{w}\t{s:.4f}")
            print("\n🔵 D (cos μ_D):")
            for w, s in out["D"]:
                print(f"{w}\t{s:.4f}")

    return out

In [20]:
res7=top_voisins_direction(df_imm, glove, collapse_lemma=True)

🧭 Mots les plus PRO-R (projection sur μ_R − μ_D) :
illegal	0.5013
illegally	0.4173
prohibited	0.3992
unlawful	0.3784
unauthorized	0.3705
illicit	0.3597
smuggling	0.3571
unlicensed	0.3546
smugglers	0.3295
profiting	0.3206

🧭 Mots les plus PRO-D (projection sur μ_R − μ_D) :
bipartisan	-0.4382
blueprint	-0.3947
balanced	-0.3809
overhaul	-0.3698
reconciliation	-0.3696
reform	-0.3692
comprehensive	-0.3598
senate	-0.3482
compromise	-0.3295
stalled	-0.3072


In [21]:
res8=top_voisins_direction(df_fed, glove, collapse_lemma=True)

🧭 Mots les plus PRO-R (projection sur μ_R − μ_D) :
billions	0.3061
infrastructure	0.2691
money	0.2559
paulson	0.2402
trillions	0.2401
costing	0.2393
reconstruction	0.2388
financial	0.2383
guaranteeing	0.2360
taxpayer	0.2281

🧭 Mots les plus PRO-D (projection sur μ_R − μ_D) :
board	-0.5678
commissioners	-0.3566
regents	-0.3551
directors	-0.3363
unanimously	-0.3264
trustees	-0.3250
excutive	-0.2801
captains	-0.2637
chairperson	-0.2485
bulletin	-0.2480


In [22]:
res9=top_voisins_direction(df_mon, glove, collapse_lemma=True)

🧭 Mots les plus PRO-R (projection sur μ_R − μ_D) :
ills	0.2788
predictability	0.2754
stability	0.2580
restoring	0.2545
capitalism	0.2516
stable	0.2499
correcting	0.2450
decent	0.2423
stabilizing	0.2394
longterm	0.2336

🧭 Mots les plus PRO-D (projection sur μ_R − μ_D) :
committee	-0.3416
senate	-0.3076
board	-0.2982
subcommittees	-0.2965
hearings	-0.2645
council	-0.2556
unanimously	-0.2473
appropriations	-0.2457
vacancies	-0.2450
appointees	-0.2411
